# QC Linked-Read Barcode Summary

In [ ]:
indir <- "logs/bxcount"
platform <- "haplotagging"

In [ ]:
IRdisplay::display_html(
    sprintf(
        "<p>%s<br>%s</p>",
        format(Sys.time(), '🗓️ %d %B, %Y 🕔 %H:%M'),
        paste("Linked-read type:", platform)
    )
)

In [ ]:
library(dplyr)
library(DT)
library(formattable)
library(highcharter)
library(scales)
library(tidyr)

In [ ]:
options(DT.options = list(class = 'table-condensed'))

In [ ]:
iframe_height <- function(x){
  if (is.data.frame(x)) {
    frameheight <- 200 + (50 * min(10, nrow(x)))
  } else {
    frameheight <- x
  }

  sprintf("
      function(el, x) {
        // Try to resize parent iframe
        if (window.frameElement) {
          window.frameElement.style.width = '100%%';
          window.frameElement.style.height = '%spx';
        }
      }
    ", frameheight)
}

trunc_digits <- function(x, digits = 0){
  x-x%%10^-digits
}

In [ ]:
create_colored_box <- function(value, label, color = "steelblue", width = "200px", height = "90px") {
  .val <- formatC(value, format = "f", big.mark = ",", digits = 2, drop0trailing = T)
  sprintf(
    '<div style="background-color: %s; width: %s; height: %s; display: flex; flex-direction: column; align-items: center; justify-content: center; color: white; border-radius: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.15); padding: 20px; box-sizing: border-box;"><div style="font-size: 14px; font-weight: normal; margin-bottom: 2px; margin-top: 13px; opacity: 0.9; text-transform: uppercase; letter-spacing: 1px;">%s</div><div style="font-size: 48px; font-weight: bold;">%s</div></div>',
    color, width, height, label, .val
    )
}

# Function to display boxes that wrap dynamically
show_boxes <- function(boxes, gap = "15px") {
  box_html <- paste(boxes, collapse = "\n")
  html_content <- sprintf('<div style="display: flex; flex-wrap: wrap; gap: %s; width: 100%%;">%s</div>', gap, box_html)
  IRdisplay::display_html(html_content)
}

This report details the counts of valid and invalid linked-read barcodes in the 
sample read headers. Validation varies between linked-read technologies, look through the tabs below
for more information on the different technologies. More exhaustive explanations can be found in 
[Harpy's documentation](https://pdimens.github.io/harpy/getting_started/linked_read_data/#linked-read-data-types).

::::{tab-set} 
:::{tab-item} Haplotagging
Segments: the `Axx`, `Bxx`, `Cxx`, `Dxx` parts of haplotagging barcodes

The correct haplotagging format is `BX:Z:AxxCxxBxxDxx`, where 
there is a tab before (and possibly after) `BX:Z` and `xx` are 2-digit numbers
between `00` and `96`. Additional tags (e.g. `QX:Z` or `VX:i`) are permissible as long as they conform to the 
[SAM comment spec (section 1.5)](https://samtools.github.io/hts-specs/SAMv1.pdf) 
of `TAG:TYPE:VALUE`.


![haplotagging FASTQ record](https://pdimens.github.io/harpy/static/lr_haplotagging.svg)
:::

:::{tab-item} stLFR
Segments: the `X`,`Y`,`Z` segments of stFLR barcodes  

The correct stLFR format is `#X_Y_Z` at the end of the sequence ID, where `X`, `Y`, and `Z` are
integers of any value, with `0` as the value for any of the segments being considered invalid (e.g. `32_0_1101`).

![stLFR FASTQ record](https://pdimens.github.io/harpy/static/lr_stlfr.svg)
:::

:::{tab-item} TELLseq
Segments: TELLseq is not combinatorial, but for the purposes of these assessments, the "segments"
are the nucleotide position in the barcode.

The correct TELLseq format is `:ATCGN` at the end of the sequence ID, where `ATCGN` are
uppercase nucleotides (`A`, `T`, `C`, `G`, `N`), with `N` being considered invalid. TELLseq
uses 18bp barcodes, so the barcodes are generally expected to be 18bp long and the presence
of `N` would make the barcode considered invalid.

![TELLseq FASTQ record](https://pdimens.github.io/harpy/static/lr_tellseq.svg)
:::
::::

In [ ]:
infiles <- list.files(indir, ".count.log", full.names = TRUE)
for (j in seq_along(infiles)) {
  i <- infiles[j]
  samplename <- gsub(".count.log", "", basename(i))
  s_df <- read.table(i, header = F)
  total <- s_df$V2[1]
  bc_total <- s_df$V2[2]
  bc_ok <- s_df$V2[3]
  bc_invalid <- s_df$V2[4]
  if (bc_total == 0) {
    bc_percent <- 0
    ig_percent <- 0
  } else {
    bc_percent <- trunc_digits((bc_ok / bc_total) * 100, 2)
    ig_percent <- trunc_digits((bc_invalid / bc_total) * 100, 2)
  }
  if(j == 1){
      BXstats <- data.frame(
        Sample = samplename,
        TotalReads = total,
        TotalBarcodes = bc_total,
        ValidBarcodes = bc_ok,
        ValidPercent = bc_percent,
        InvalidBarcodes = bc_invalid,
        InvalidPercent = ig_percent
      )
      .invalid <- t(rbind(c("Sample", samplename), c("InvalidBarcodes", bc_invalid),s_df[5:nrow(s_df),]))
      invalids <- data.frame(t(.invalid[2,]))
      names(invalids) <- .invalid[1,]
      invalids <- invalids %>% mutate_at(-1, as.numeric)
  } else {
    BXstats <- rbind(
      BXstats,
      data.frame(
        Sample = samplename,
        TotalReads = total,
        TotalBarcodes = bc_total,
        ValidBarcodes = bc_ok,
        ValidPercent = bc_percent,
        InvalidBarcodes = bc_invalid,
        InvalidPercent = ig_percent
      )
    )
    .invalid <- t(rbind(c("Sample", samplename), c("InvalidBarcodes", bc_invalid), s_df[5:nrow(s_df),]))
    .inv <- data.frame(t(.invalid[2,]))
    names(.inv) <- .invalid[1,]
    .inv <- .inv %>% mutate_at(2:ncol(.inv), as.numeric)
    invalids <- rbind(invalids, .inv)
    }
}

In [ ]:
invalids_percent <- invalids %>% mutate_at(c(-1,-2), function(x){trunc_digits(x/(.$InvalidBarcodes) * 100, 1)})

.stats <- merge(BXstats[,1:5], invalids_percent[,-2])

invalids_long <- pivot_longer(invalids_percent, -1, names_to = "Position", values_to = "Count")
invalids_long$Position <- as.factor(paste0("position " , invalids_long$Position))

In [ ]:
position_names <- names(invalids)[3:ncol(invalids)]
# a 256-color distribution of the Turbo palette
# from https://github.com/dbaranger/turbo_palette_R/blob/main/turbo_hex.txt
palette <- c("#30123B","#321543","#33184A","#341B51","#351E58","#36215F","#372466","#38276D","#392A73","#3A2D79","#3B2F80","#3C3286","#3D358B","#3E3891","#3F3B97","#3F3E9C","#4040A2","#4143A7","#4146AC","#4249B1","#424BB5","#434EBA","#4451BF","#4454C3","#4456C7","#4559CB","#455CCF","#455ED3","#4661D6","#4664DA","#4666DD","#4669E0","#466BE3","#476EE6","#4771E9","#4773EB","#4776EE","#4778F0","#477BF2","#467DF4","#4680F6","#4682F8","#4685FA","#4687FB","#458AFC","#458CFD","#448FFE","#4391FE","#4294FF","#4196FF","#4099FF","#3E9BFE","#3D9EFE","#3BA0FD","#3AA3FC","#38A5FB","#37A8FA","#35ABF8","#33ADF7","#31AFF5","#2FB2F4","#2EB4F2","#2CB7F0","#2AB9EE","#28BCEB","#27BEE9","#25C0E7","#23C3E4","#22C5E2","#20C7DF","#1FC9DD","#1ECBDA","#1CCDD8","#1BD0D5","#1AD2D2","#1AD4D0","#19D5CD","#18D7CA","#18D9C8","#18DBC5","#18DDC2","#18DEC0","#18E0BD","#19E2BB","#19E3B9","#1AE4B6","#1CE6B4","#1DE7B2","#1FE9AF","#20EAAC","#22EBAA","#25ECA7","#27EEA4","#2AEFA1","#2CF09E","#2FF19B","#32F298","#35F394","#38F491","#3CF58E","#3FF68A","#43F787","#46F884","#4AF880","#4EF97D","#52FA7A","#55FA76","#59FB73","#5DFC6F","#61FC6C","#65FD69","#69FD66","#6DFE62","#71FE5F","#75FE5C","#79FE59","#7DFF56","#80FF53","#84FF51","#88FF4E","#8BFF4B","#8FFF49","#92FF47","#96FE44","#99FE42","#9CFE40","#9FFD3F","#A1FD3D","#A4FC3C","#A7FC3A","#A9FB39","#ACFB38","#AFFA37","#B1F936","#B4F836","#B7F735","#B9F635","#BCF534","#BEF434","#C1F334","#C3F134","#C6F034","#C8EF34","#CBED34","#CDEC34","#D0EA34","#D2E935","#D4E735","#D7E535","#D9E436","#DBE236","#DDE037","#DFDF37","#E1DD37","#E3DB38","#E5D938","#E7D739","#E9D539","#EBD339","#ECD13A","#EECF3A","#EFCD3A","#F1CB3A","#F2C93A","#F4C73A","#F5C53A","#F6C33A","#F7C13A","#F8BE39","#F9BC39","#FABA39","#FBB838","#FBB637","#FCB336","#FCB136","#FDAE35","#FDAC34","#FEA933","#FEA732","#FEA431","#FEA130","#FE9E2F","#FE9B2D","#FE992C","#FE962B","#FE932A","#FE9029","#FD8D27","#FD8A26","#FC8725","#FC8423","#FB8122","#FB7E21","#FA7B1F","#F9781E","#F9751D","#F8721C","#F76F1A","#F66C19","#F56918","#F46617","#F36315","#F26014","#F15D13","#F05B12","#EF5811","#ED5510","#EC530F","#EB500E","#EA4E0D","#E84B0C","#E7490C","#E5470B","#E4450A","#E2430A","#E14109","#DF3F08","#DD3D08","#DC3B07","#DA3907","#D83706","#D63506","#D43305","#D23105","#D02F05","#CE2D04","#CC2B04","#CA2A04","#C82803","#C52603","#C32503","#C12302","#BE2102","#BC2002","#B91E02","#B71D02","#B41B01","#B21A01","#AF1801","#AC1701","#A91601","#A71401","#A41301","#A11201","#9E1001","#9B0F01","#980E01","#950D01","#920B01","#8E0A01","#8B0902","#880802","#850702","#810602","#7E0502","#7A0403") 
selected_colors <- palette[seq(1,length(palette),length(palette)%/%(length(position_names)-1))]

In [ ]:
show_boxes(c(
  create_colored_box(nrow(BXstats), "Files", "#aeaeaeff"),
  create_colored_box(min(BXstats$ValidPercent), "Min %Valid", ifelse(min(BXstats$ValidPercent) < 30, "#f6ab3c", "#68ae6bff")),
  create_colored_box(max(BXstats$ValidPercent), "Max %Valid", ifelse(max(BXstats$ValidPercent) < 75, "#f6ab3c", "#68ae6bff")),
  create_colored_box(mean(BXstats$ValidPercent), "Mean %Valid", ifelse(mean(BXstats$ValidPercent) < 50, "#f6ab3c", "#68ae6bff"))
))

## Barcode Statistics

Below is a table[^1] listing all the samples Harpy processed and their associated
barcode statistics as determined by the sequences in the **forward-read file (R1) only**.
If for some reason `TotalBarcodes` equals `0`, then there may be an issue with
the format of your FASTQ headers.

[^1]:
    | Column Name | Description |
    |:------------|:------------|
    | Sample  | name of the sample |
    | TotalReads  | total number of reads |
    |  TotalBarcodes | total number of barcodes present |
    |  ValidBarcodes | number of valid linked-read barcodes |
    |  ValidPercent | percent of linked-read barcodes that are valid for that chemistry |
    |  ... | percent invalidations at this barcode segment/position |


In [ ]:
plotdata <- BXstats %>%
  select(Sample, ValidBarcodes, InvalidBarcodes) %>% 
  rename(Valid = ValidBarcodes, Invalid = InvalidBarcodes)

percdata <- BXstats %>%
  select(Sample, ValidPercent, InvalidPercent) %>% 
  rename(Valid= ValidPercent, Invalid = InvalidPercent)

In [ ]:
as.datatable(
  formattable(.stats, list(ValidPercent = proportion_bar("lightblue"))),
  rownames = FALSE,
  extensions = c("Buttons", "FixedColumns"),
  fillContainer = T,
  options = list(
    dom = "Brtp",
    buttons = c("csv"),
    scrollY = F,
    deferRender = T,
    scrollX = T,
    fixedColumns = list(leftColumns = 1)
  )
) %>%
  htmlwidgets::onRender(iframe_height(.stats))

In [ ]:
hs <- hist(percdata$Valid, breaks = seq(0, 100, 5), plot = F)
hs <- data.frame(val = hs$breaks[-1], freq = hs$counts)

highchart(height = 400) |>
  hc_add_theme(hc_theme_elementary()) |>
  hc_chart(type = "area", animation = F) |>
  hc_add_series(data = hs, type = "areaspline", hcaes(x = val, y = freq), color = "#90cb8cff", name = "Valid Barcodes", marker = list(enabled = FALSE)) |>
  hc_title(text = "Distribution of Percent Valid Barcodes") |>
  hc_xAxis(title = list(text = "Percent Valid"), type = "linear") |>
  hc_yAxis(title = list(text = "Number of Samples")) |>
  hc_legend(enabled = FALSE) |>
  hc_tooltip(crosshairs = TRUE, animation = FALSE,
    formatter = JS("function () {return '<b>' + this.series.name + '</b><br><b>' + this.x + '% valid</b><br/>Samples: <b>' + this.y + '</b>';}") 
  ) |>
  hc_exporting(enabled = T, filename = "valid.barcodes.distribution",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
) %>% htmlwidgets::onRender(iframe_height(418))

## Frequency of Invalid Positions
This figure shows the distribution of invalid barcode positions
across all samples as a percent of all the invalid barcodes. This should help you assess whether some positions have a 
higher incidence of failure than others.

In [ ]:
percent_breaks <- seq(0, 100, 5)
inv_hist <- data.frame(Position = character(), Bin = numeric(), Count = numeric())
for(pos in position_names){
  h <- hist(invalids_percent[,pos], breaks = percent_breaks, plot = F)
  inv_hist <- rbind(
    inv_hist,
    data.frame(Position = rep(paste("position", pos), length(percent_breaks)-1), Bin = h$breaks[-1], Count = h$count)
  )
}

highchart(height = 400) |>
  hc_add_series(data = inv_hist, type = "areaspline", hcaes(x = Bin, y = Count, group = Position), stack = "overlap", marker = list(enabled = FALSE)) |>
  hc_add_theme(hc_theme_elementary()) |>
  hc_colors(selected_colors) |>
  hc_title(text = "Distribution of Invalid Barcode Positions") |>
  hc_xAxis(title = list(text = "Percent Invalid"), type = "linear", max = 100) |>
  hc_yAxis(title = list(text = "Number of Samples")) |>
  hc_tooltip(crosshairs = TRUE, animation = FALSE,
    formatter = JS("function () {return '<b>' + this.series.name + '</b><br><b>' + this.x + '% invalid</b><br/>Samples: <b>' + this.y + '</b>';}") 
  ) |>
  hc_legend(layout = "vertical", align = "right", verticalAlign = "middle") |>
  hc_exporting(enabled = T, filename = "invalid.barcodes.distribution",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
) %>% htmlwidgets::onRender(iframe_height(418))